# Poseidon Finetuning on Colab (T4)

Runs Unsloth QLoRA on the 22.5K phishing detection dataset, exports GGUF.

**Runtime:** T4 GPU, High-RAM

**Estimated time:** ~1-2 hours for 1-2 epochs

In [ ]:
# Install dependencies
!pip install unsloth==2026.5.2 unsloth_zoo==2026.5.1 && pip install xformers==0.0.29 --index-url https://download.pytorch.org/whl/cu121
import os
os.environ['UNSLOTH_RETURN_LOGITS'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import torch
print(f'GPU: {torch.cuda.get_device_name()}, VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f}GB')

In [ ]:
# Download dataset
# Will be replaced with actual URL when dataset is ready
DATASET_URL = "https://transfer.sh/poseidon_dataset.jsonl"  # REPLACE ME
!wget -O /content/dataset.jsonl $DATASET_URL
!wc -l /content/dataset.jsonl

In [ ]:
# Imports
import json, sys
from pathlib import Path
from dataclasses import dataclass
import unsloth
from datasets import Dataset
from transformers import DataCollatorForSeq2Seq
from trl import SFTTrainer, SFTConfig
from unsloth import FastLanguageModel, is_bfloat16_supported

In [ ]:
# Config
@dataclass
class Config:
    dataset: str = "/content/dataset.jsonl"
    output_dir: str = "/content/models/finetuned"
    model: str = "unsloth/gemma-3-1b-it-bnb-4bit"
    epochs: int = 2
    lr: float = 2e-4
    batch_size: int = 8
    grad_accum: int = 2
    max_len: int = 768
    lora_r: int = 16
    lora_alpha: int = 16
    lora_dropout: float = 0.0
    logging_steps: int = 50
    quant: str = "4bit"
    skip_gguf: bool = False

config = Config()
out = Path(config.output_dir)
out.mkdir(parents=True, exist_ok=True)

In [ ]:
# Load dataset
rows = []
with open(config.dataset) as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))
print(f"Loaded {len(rows)} rows")

conversations = []
for row in rows:
    prompt = row.get("prompt", "")
    response = row.get("assistant_raw", "")
    if prompt and response:
        conversations.append({"messages": [
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": response},
        ]})
print(f"Conversations: {len(conversations)}")

In [ ]:
# Load model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=config.model,
    max_seq_length=config.max_len,
    dtype=None,
    load_in_4bit=(config.quant == "4bit"),
    load_in_8bit=(config.quant == "8bit"),
    device_map="auto",
)

# Apply LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=config.lora_r,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=config.lora_alpha,
    lora_dropout=config.lora_dropout,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

In [ ]:
# Tokenize
response_token_ids = tokenizer.encode("<start_of_turn>model\n", add_special_tokens=False)

def fmt(messages):
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

formatted = [fmt(ex["messages"]) for ex in conversations]

input_ids_list, labels_list = [], []
skipped = 0
for text in formatted:
    tokens = tokenizer.encode(text, add_special_tokens=False)
    resp_start = None
    for i in range(len(tokens)):
        if tokens[i:i+len(response_token_ids)] == response_token_ids:
            resp_start = i
            break
    if resp_start is None:
        skipped += 1
        continue
    if len(tokens) > config.max_len:
        resp = tokens[resp_start:]
        user = tokens[:resp_start]
        budget = config.max_len - len(resp)
        if budget <= 0:
            skipped += 1
            continue
        tokens = user[-budget:] + resp
        resp_start = budget
    labels = [-100] * len(tokens)
    for j in range(resp_start, len(tokens)):
        labels[j] = tokens[j]
    input_ids_list.append(tokens)
    labels_list.append(labels)

print(f"Tokenized {len(input_ids_list)} examples ({skipped} skipped)")

tokenized_dataset = Dataset.from_dict({
    "input_ids": input_ids_list,
    "labels": labels_list,
    "attention_mask": [[1]*len(ids) for ids in input_ids_list],
})

In [ ]:
# Train
training_args = SFTConfig(
    output_dir=str(out / "checkpoints"),
    num_train_epochs=config.epochs,
    per_device_train_batch_size=config.batch_size,
    gradient_accumulation_steps=config.grad_accum,
    warmup_steps=10,
    logging_steps=config.logging_steps,
    learning_rate=config.lr,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    save_strategy="epoch",
    report_to="none",
    remove_unused_columns=True,
    dataloader_num_workers=2,
    max_length=config.max_len,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=tokenized_dataset,
    args=training_args,
)

print(f"Starting {config.epochs} epoch(s)...")
trainer.train()

In [ ]:
# Export to GGUF
gguf_path = out / "poseidon-phishing-detect-q4_k_m.gguf"
print(f"Exporting GGUF...")
from unsloth.save import unsloth_save_pretrained_gguf
unsloth_save_pretrained_gguf(
    trainer.model,
    save_directory=str(gguf_path),
    tokenizer=tokenizer,
    quantization_method="q4_k_m",
)
print(f"GGUF: {gguf_path}")
import os; print(f"Size: {os.path.getsize(gguf_path) / 1e6:.0f} MB")

In [ ]:
# Download GGUF to local machine
from google.colab import files
files.download(str(gguf_path))